# Clase 08: Análisis Discriminante Lineal (LDA)

**Docente:** Josef Rodriguez  
**Duración estimada:** 2 horas  
**Nivel:** Avanzado

---

## Objetivos de aprendizaje

Al finalizar esta clase, el estudiante podrá:

1. Entender la intuición geométrica detrás de la regla de Fisher.
2. Calcular manualmente la dirección óptima de proyección.
3. Interpretar la frontera de decisión en LDA.
4. Extender el análisis al caso multiclase.
5. Comparar LDA con regresión logística.
6. Comprender el efecto de los outliers sobre ambos enfoques.


## 1. Librerías

En esta sección importamos las librerías necesarias para construir ejemplos, visualizar la geometría de LDA y comparar con regresión logística.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

np.random.seed(42)


## 2. Generación del dataset base

Creamos un problema binario simple con dos nubes de puntos. Esto nos permitirá visualizar claramente la idea de proyección discriminante.


In [ ]:
X, y = make_blobs(
    n_samples=250,
    centers=[[-1.5, -1.0], [1.8, 1.5]],
    cluster_std=[1.2, 1.0],
    random_state=42
)

X0 = X[y == 0]
X1 = X[y == 1]

print('Forma de X:', X.shape)
print('Distribución de clases:')
print(pd.Series(y).value_counts().sort_index())


## 3. Visualización inicial

Primero observamos el problema en el espacio original. La idea central es responder una pregunta:

**¿En qué dirección conviene mirar los datos para separar mejor ambas clases?**


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X0[:, 0], X0[:, 1], alpha=0.7, label='Clase 0')
plt.scatter(X1[:, 0], X1[:, 1], alpha=0.7, label='Clase 1')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Dos clases en el espacio original')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 4. Direcciones posibles de proyección

No toda dirección separa igual. Antes de llegar a la solución de Fisher, visualizamos varias direcciones candidatas.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X0[:, 0], X0[:, 1], alpha=0.6, label='Clase 0')
plt.scatter(X1[:, 0], X1[:, 1], alpha=0.6, label='Clase 1')

origin = X.mean(axis=0)

directions = {
    'Direccion 1': np.array([1.0, 0.2]),
    'Direccion 2': np.array([0.7, 0.7]),
    'Direccion 3': np.array([0.5, 1.0])
}

for name, d in directions.items():
    d = d / np.linalg.norm(d)
    plt.arrow(
        origin[0], origin[1], 2*d[0], 2*d[1],
        head_width=0.15, length_includes_head=True
    )

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('No todas las direcciones separan igual')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 5. Regla de Fisher: cálculo manual

La idea de Fisher es encontrar la dirección $w$ que maximiza la razón entre:

- separación entre clases
- dispersión dentro de las clases

En el caso binario, la solución cerrada es:

$$
w = S_W^{-1}(\mu_1 - \mu_0)
$$

donde:

- $\mu_0, \mu_1$ son las medias de cada clase
- $S_W$ es la matriz de dispersión dentro de clase


In [ ]:
mu0 = X0.mean(axis=0)
mu1 = X1.mean(axis=0)

Sw = (X0 - mu0).T @ (X0 - mu0) + (X1 - mu1).T @ (X1 - mu1)
w = np.linalg.inv(Sw) @ (mu1 - mu0)
w = w / np.linalg.norm(w)

print('Media clase 0:', mu0)
print('Media clase 1:', mu1)
print('Direccion de Fisher w:', w)


## 6. Visualización de la dirección óptima

Ahora dibujamos la dirección óptima de Fisher sobre el espacio original.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X0[:, 0], X0[:, 1], alpha=0.6, label='Clase 0')
plt.scatter(X1[:, 0], X1[:, 1], alpha=0.6, label='Clase 1')
plt.scatter(*mu0, s=180, marker='X', label='Centro clase 0')
plt.scatter(*mu1, s=180, marker='X', label='Centro clase 1')

mid = (mu0 + mu1) / 2
plt.arrow(
    mid[0], mid[1], 2.5*w[0], 2.5*w[1],
    head_width=0.15, length_includes_head=True
)

plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Dirección óptima de Fisher')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 7. Proyección de los datos en una dimensión

Una vez obtenida la dirección $w$, proyectamos cada observación sobre dicha recta:

$$
z = Xw
$$

Esto reduce el problema de dos dimensiones a una dimensión.


In [ ]:
z0 = X0 @ w
z1 = X1 @ w

plt.figure(figsize=(8, 5))
plt.hist(z0, bins=20, alpha=0.6, density=True, label='Clase 0')
plt.hist(z1, bins=20, alpha=0.6, density=True, label='Clase 1')
plt.xlabel('Proyección sobre w')
plt.ylabel('Densidad')
plt.title('Proyección 1D sobre la dirección de Fisher')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 8. Umbral de decisión

Si ya tenemos los puntos proyectados, clasificar es equivalente a definir un umbral. Un criterio simple es usar el punto medio entre las medias proyectadas:

$$
th = \frac{\bar z_0 + \bar z_1}{2}
$$


In [ ]:
th = (z0.mean() + z1.mean()) / 2

plt.figure(figsize=(8, 5))
plt.hist(z0, bins=20, alpha=0.6, density=True, label='Clase 0')
plt.hist(z1, bins=20, alpha=0.6, density=True, label='Clase 1')
plt.axvline(th, linestyle='--', linewidth=2, label='Umbral')
plt.xlabel('Proyección sobre w')
plt.ylabel('Densidad')
plt.title('Regla de decisión en una dimensión')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print('Umbral de decisión:', th)


## 9. LDA con scikit-learn

Ahora comparamos el cálculo manual con la implementación estándar de `scikit-learn`.


In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X, y)

print('Coeficientes LDA:')
print(lda.coef_)
print('Intercepto:')
print(lda.intercept_)


## 10. Predicción y métricas del modelo LDA

Evaluamos el desempeño del modelo entrenado sobre el mismo conjunto de datos con fines didácticos.


In [ ]:
y_pred_lda = lda.predict(X)
y_prob_lda = lda.predict_proba(X)[:, 1]

acc_lda = accuracy_score(y, y_pred_lda)
auc_lda = roc_auc_score(y, y_prob_lda)

print(f'Accuracy LDA: {acc_lda:.4f}')
print(f'AUC LDA: {auc_lda:.4f}')


## 11. LDA multiclase

Una gran ventaja de LDA es que se generaliza naturalmente al caso multiclase. Si tenemos $K$ clases, el espacio discriminante tiene como máximo $K-1$ dimensiones.


In [ ]:
X3, y3 = make_blobs(
    n_samples=300,
    centers=[[-2, -1], [2, 2], [0, 4]],
    cluster_std=[1.2, 1.0, 1.1],
    random_state=42
)

lda_mc = LinearDiscriminantAnalysis(n_components=2)
X3_lda = lda_mc.fit_transform(X3, y3)

plt.figure(figsize=(8, 6))
for cls in np.unique(y3):
    plt.scatter(X3_lda[y3 == cls, 0], X3_lda[y3 == cls, 1], alpha=0.7, label=f'Clase {cls}')

plt.xlabel('LD1')
plt.ylabel('LD2')
plt.title('Proyección multiclase en el espacio discriminante')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 12. Comparación entre LDA y regresión logística

Comparamos dos enfoques lineales:

- **LDA**: generativo, con supuestos de normalidad multivariante y covarianza común.
- **Regresión logística**: discriminativa, más flexible frente a violaciones de supuestos.


In [ ]:
lda_cmp = LinearDiscriminantAnalysis()
log_cmp = LogisticRegression(C=1e6, max_iter=1000)

lda_cmp.fit(X, y)
log_cmp.fit(X, y)

auc_lda_cmp = roc_auc_score(y, lda_cmp.predict_proba(X)[:, 1])
auc_log_cmp = roc_auc_score(y, log_cmp.predict_proba(X)[:, 1])

acc_lda_cmp = accuracy_score(y, lda_cmp.predict(X))
acc_log_cmp = accuracy_score(y, log_cmp.predict(X))

print(f'LDA -> AUC: {auc_lda_cmp:.4f} | Accuracy: {acc_lda_cmp:.4f}')
print(f'Logística -> AUC: {auc_log_cmp:.4f} | Accuracy: {acc_log_cmp:.4f}')


## 13. Experimento con outliers

Ahora contaminamos el dataset con observaciones extremas para observar la sensibilidad de ambos modelos.

Esto es importante porque LDA depende de medias y matrices de covarianza, por lo que suele ser más sensible a valores atípicos.


In [ ]:
rng = np.random.RandomState(42)
outliers = rng.uniform(low=-8, high=8, size=(30, 2))
y_out = rng.randint(0, 2, size=30)

X_out = np.vstack([X, outliers])
y_out_full = np.concatenate([y, y_out])

plt.figure(figsize=(8, 6))
plt.scatter(X_out[y_out_full == 0, 0], X_out[y_out_full == 0, 1], alpha=0.6, label='Clase 0')
plt.scatter(X_out[y_out_full == 1, 0], X_out[y_out_full == 1, 1], alpha=0.6, label='Clase 1')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Dataset con outliers')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


In [ ]:
lda_out = LinearDiscriminantAnalysis()
log_out = LogisticRegression(C=1e6, max_iter=1000)

lda_out.fit(X_out, y_out_full)
log_out.fit(X_out, y_out_full)

auc_lda_out = roc_auc_score(y_out_full, lda_out.predict_proba(X_out)[:, 1])
auc_log_out = roc_auc_score(y_out_full, log_out.predict_proba(X_out)[:, 1])

acc_lda_out = accuracy_score(y_out_full, lda_out.predict(X_out))
acc_log_out = accuracy_score(y_out_full, log_out.predict(X_out))

print(f'LDA con outliers -> AUC: {auc_lda_out:.4f} | Accuracy: {acc_lda_out:.4f}')
print(f'Logística con outliers -> AUC: {auc_log_out:.4f} | Accuracy: {acc_log_out:.4f}')


## 14. Conclusiones

Ideas clave de la sesión:

1. LDA busca la mejor proyección lineal para separar clases.
2. La regla de Fisher maximiza separación entre medias y minimiza dispersión interna.
3. En el caso binario, la solución cerrada es $w = S_W^{-1}(\mu_1 - \mu_0)$.
4. LDA también sirve como reducción de dimensión supervisada.
5. Cuando se cumplen los supuestos, LDA puede ser muy eficiente.
6. En presencia de outliers o violaciones fuertes de supuestos, regresión logística suele ser más robusta.
